# Evaluation – Step 2: LLM-as-a-Judge

Runs the agent against test questions (from `questions.json`),
then evaluates each answer with a Groq judge model.
Results are saved to a Pandas DataFrame and printed as summary statistics.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'app'))

import json, asyncio
from pathlib import Path
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel
from tqdm.auto import tqdm
import pandas as pd

import ingest, search_agent, logs

In [ ]:
# ── Load data & initialise agent ───────────────────────────────────────────
index, _ = ingest.index_data(
    'evidentlyai', 'docs',
    chunk=True,
    chunking_params={'method': 'sliding_window', 'size': 2000, 'step': 1000},
)
agent = search_agent.init_agent(index)

In [ ]:
# ── Load test questions ────────────────────────────────────────────────────
questions = json.loads(Path('questions.json').read_text())
print(f'{len(questions)} questions loaded.')

In [ ]:
# ── Run agent on all questions and log results ─────────────────────────────
for q in tqdm(questions):
    result = await agent.run(user_prompt=q)
    logs.log_interaction_to_file(agent, result.new_messages(), source='ai-generated')
print('Done.')

In [ ]:
# ── Evaluation agent ───────────────────────────────────────────────────────
EVAL_PROMPT = """
Use this checklist to evaluate the quality of an AI agent's answer to a user question.

Checklist:
- instructions_follow: Agent followed its system instructions.
- answer_relevant: Response directly addresses the question.
- answer_clear: Answer is clear and factually reasonable.
- answer_citations: Response includes proper source citations.
- completeness: Response covers all key aspects of the question.
- tool_call_search: The search tool was invoked.

Output true/false for each check with a short justification.
""".strip()

class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str

eval_agent = Agent(
    name='eval_agent',
    model=GroqModel('llama-3.3-70b-versatile'),
    instructions=EVAL_PROMPT,
    output_type=EvaluationChecklist,
)

USER_PROMPT_FMT = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

In [ ]:
def simplify_log(messages):
    simplified = []
    for m in messages:
        parts = []
        for p in m['parts']:
            cp = p.copy()
            if cp.get('part_kind') == 'tool-return':
                cp['content'] = 'REDACTED'
            cp.pop('timestamp', None)
            cp.pop('tool_call_id', None)
            cp.pop('metadata', None)
            cp.pop('id', None)
            parts.append(cp)
        simplified.append({'kind': m['kind'], 'parts': parts})
    return simplified

async def evaluate_log(log_record):
    msgs = log_record['messages']
    question = msgs[0]['parts'][0]['content']
    answer = msgs[-1]['parts'][0]['content']
    log_str = json.dumps(simplify_log(msgs))
    prompt = USER_PROMPT_FMT.format(
        instructions=log_record['system_prompt'],
        question=question,
        answer=answer,
        log=log_str,
    )
    result = await eval_agent.run(prompt)
    return question, answer, result.output

In [ ]:
# ── Run evaluations ────────────────────────────────────────────────────────
log_files = logs.list_log_files(agent_name='evidently_agent', source='ai-generated')
print(f'Evaluating {len(log_files)} log files …')

rows = []
for lf in tqdm(log_files):
    record = logs.load_log_file(lf)
    question, answer, checklist = await evaluate_log(record)
    row = {'question': question, 'answer': answer}
    row.update({c.check_name: c.check_pass for c in checklist.checklist})
    row['summary'] = checklist.summary
    rows.append(row)

df = pd.DataFrame(rows)
df

In [ ]:
# ── Summary statistics ─────────────────────────────────────────────────────
print('=== Pass-rate per check ===')
print(df.select_dtypes(include='bool').mean().round(2).to_string())

# Save to CSV for the README
df.to_csv('eval_results.csv', index=False)
print('\nSaved → eval_results.csv')